<a href="https://colab.research.google.com/github/scostavinicius/lean-agent/blob/main/lean_agent_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assistente de provas em Lean 4 — grafo LangGraph

**Dupla:** VINICIUS COSTA SOARES · MATHEUS VINÍCIUS SILVA FREIRE DE CASTRO

Tarefa da Unidade II — IA Agêntica (2026.2)

O código do agente vive no pacote `lean_agent/` do repositório; este notebook só o usa. O estado, os nós e as arestas do grafo estão descritos no README.

**Onde rodar:**
- **Colab:** a primeira célula clona o repositório na branch de `BRANCH`. A branch precisa estar no GitHub (`git push`).
- **Local (VS Code ou JupyterLab):** defina a variável de ambiente `GROQ_API_KEY` antes de abrir o editor, abra o notebook da raiz do repositório e escolha o Python do `.venv` como kernel. Nada é clonado, e o Lean instalado pelo elan é reaproveitado. O passo a passo está no README.

## 1 — Ambiente

In [ ]:
# Por quê: no Colab o repositório ainda não está na máquina e é clonado aqui.
# Rodando local, da raiz do repositório, nada é clonado. O %pip instala no
# Python do kernel, que é onde os imports abaixo procuram.
import os
import subprocess
import sys

REPOSITORIO = "https://github.com/scostavinicius/lean-agent.git"
BRANCH = "feat/langgraph"  # depois do merge, troque para "main"

if not os.path.exists("lean_agent"):
    if not os.path.exists("lean-agent"):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPOSITORIO], check=True)
    os.chdir("lean-agent")
%pip install -q -r requirements.txt
print("pasta do projeto:", os.getcwd())

In [ ]:
# Por quê: o Lean é o verificador do agente. Se ele já estiver instalado pelo
# elan (como na máquina da dupla), nada é baixado. No Colab, o elan instala uma
# versão fixa. No Windows o instalador é outro, por isso só o aviso.
import platform
import shutil

os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + os.pathsep + os.environ["PATH"]
if shutil.which("lean") is None:
    if platform.system() == "Windows":
        raise RuntimeError("Lean não encontrado: instale pelo elan (https://github.com/leanprover/elan)")
    subprocess.run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh"
                   " | sh -s -- -y --default-toolchain leanprover/lean4:v4.33.0", shell=True, check=True)
print(subprocess.run(["lean", "--version"], capture_output=True, text=True).stdout)

In [ ]:
# Por quê: a tarefa proíbe a chave no repositório e no notebook. Ela é procurada,
# nesta ordem: variável de ambiente GROQ_API_KEY (o jeito de rodar local), segredo
# do Colab, arquivo .env na raiz do repositório e, por último, digitada aqui.
# Só a origem é impressa, nunca a chave.
from dotenv import load_dotenv


def carregar_chave() -> str:
    """Põe a GROQ_API_KEY em os.environ e devolve de onde ela veio."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "segredo do Colab"
    except Exception:
        pass  # fora do Colab, ou sem o segredo cadastrado
    if load_dotenv() and os.environ.get("GROQ_API_KEY"):
        return "arquivo .env"
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    return "digitada no notebook"


print("GROQ_API_KEY lida de:", carregar_chave())

## 2 — O grafo

In [ ]:
# Por quê: o desenho sai do grafo compilado, então é sempre o grafo que roda.
from IPython.display import Image, Markdown, display

from lean_agent import criar_grafo, provar

grafo = criar_grafo()
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception as erro:  # o PNG usa o serviço mermaid.ink; sem rede, fica o texto
    print(f"sem PNG ({type(erro).__name__}); diagrama em Mermaid:")
    print(grafo.get_graph().draw_mermaid())

In [ ]:
# Por quê: o rastro mostra cada ferramenta que o agente escolheu e o que o Lean
# respondeu. É a evidência de que a decisão é dele, e de que quem encerra é o Lean.
# Troque NOME para ver o rastro de outro caso.
from lean_agent.avaliacao import mostrar_rastro
from lean_agent.casos import CASOS

NOME = "TESTE1"

caso = next(c for c in CASOS if c["nome"] == NOME)
estado = provar(grafo, caso["enunciado"], caso["nome"])
mostrar_rastro(estado)
print("\n--- resposta ao usuário ---")
print(estado["messages"][-1].content)

In [ ]:
# Por quê: memória com checkpointer. Os dois turnos usam a mesma thread, e o
# segundo só faz sentido se o assistente lembrar do primeiro. O primeiro pedido
# está em português: o nó interpretar precisa formalizá-lo em Lean.
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "demo-memoria"}}
for texto in ["Prove que p ∧ q implica q ∧ p, para proposições p e q.",
              "Qual tática você usou na prova anterior?"]:
    estado = grafo.invoke({"messages": [HumanMessage(texto)]}, config)
    print(f"você> {texto}\nassistente> {estado['messages'][-1].content}\n")

## 3 — Testes automáticos (sem API)

Parser do log do Lean, provas de referência dos 10 casos e as arestas do grafo com um modelo roteirizado. Não gastam tokens da Groq.

In [ ]:
# Por quê: é o mesmo `pytest` da linha de comando, num processo separado para
# não misturar o estado do notebook com o dos testes.
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:warnings"],
                           capture_output=True, text=True, encoding="utf-8", errors="replace",
                           env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(resultado.stdout[-3000:] + resultado.stderr[-3000:])

## 4 — Casos de teste do agente

Cada caso passa pelo grafo inteiro, com a Groq e o Lean. Os 10 juntos gastam boa parte do limite diário do plano gratuito (200 mil tokens): para experimentar, ponha poucos nomes em `CASOS_ESCOLHIDOS`.

In [ ]:
# Por quê: rodar os casos pelo grafo e montar a tabela, com quanto cada um custou.
# Nos casos "falso", acertar é o Lean NÃO aceitar prova nenhuma.
# CASOS_ESCOLHIDOS vazio roda os 10; com nomes, roda só esses e mostra o rastro.
from lean_agent.avaliacao import rodar_casos

CASOS_ESCOLHIDOS = []  # ex.: ["TESTE1", "TESTE7"]
PAUSA = 20             # segundos entre casos, para folgar o limite por minuto da Groq

resultados = rodar_casos(CASOS_ESCOLHIDOS, pausa=PAUSA, rastro=bool(CASOS_ESCOLHIDOS), grafo=grafo)

In [ ]:
# Por quê: a tabela no formato que o README pede (entrada, esperado, obtido e se
# foi aprovado). O texto impresso embaixo é o Markdown, pronto para copiar.
from lean_agent.avaliacao import tabela_markdown

tabela = tabela_markdown(resultados)
display(Markdown(tabela))
print(tabela)